# LoRA and Efficient Fine-Tuning

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 08.05 |
| Time | ~85 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/08_Advanced_Linear_Algebra/06_lora_and_efficient_finetuning.ipynb)

---

## 🎯 Learning Objective

Derive the LoRA update $W = W_0 + BA$ from low-rank approximation theory, and understand rank choice and QLoRA.

---

## 📐 Theory

Every idea this module needed is now in place: Gram matrices and projection
([08.01](01_inner_product_spaces.ipynb)), matrix-shaped gradients
([08.02](02_matrix_calculus.ipynb)), and — most directly — the Eckart-Young theorem and
randomized low-rank approximation ([08.05](05_low_rank_approximations.ipynb)), itself built on
truncated SVD from [01.07](../01_Linear_Algebra/07_SVD_and_PCA.ipynb). **LoRA (Low-Rank
Adaptation)**, introduced by Hu et al. (2021), is what happens when you stop using low-rank
approximation to *compress an already-trained* matrix and start using it to *define what gets
trained in the first place*.

### The core idea: freeze $W_0$, learn a low-rank update

Fine-tuning a pretrained weight matrix $W_0\in\mathbb{R}^{d\times k}$ normally means updating
every one of its $dk$ entries. LoRA instead freezes $W_0$ completely and represents the *update*
as a product of two small matrices:

$$W = W_0 + \Delta W, \qquad \Delta W = BA, \qquad B\in\mathbb{R}^{d\times r},\ \ A\in\mathbb{R}^{r\times k},\ \ r \ll \min(d,k)$$

Only $B$ and $A$ are trained; $W_0$ never changes. This is a **rank constraint imposed on the
update itself**, not on $W$: $W$ can still be full rank (since $W_0$ is), but whatever *changed*
about it during fine-tuning is restricted to rank $\le r$. The parameter count drops from $dk$
to $r(d+k)$ — for $r\ll d,k$, a reduction of roughly $dk/(r(d+k)) \approx d/(2r)$ when $d\approx
k$, which is why $r$ in the tens against $d$ in the thousands routinely gives $100$-$1000\times$
fewer trainable parameters.

**Initialization matters**: $A$ is initialized random (small values), but $B$ is initialized to
exactly **zero**. Since $\Delta W = BA$, this makes $\Delta W = 0$ at the very start of
training — the adapted model computes *identically* to the frozen base model before any
training happens, so LoRA can never make a pretrained model worse before you've even started
fine-tuning it.

### Why is this a reasonable thing to do at all?

This is where [08.05](05_low_rank_approximations.ipynb) stops being background and becomes the
actual justification. The Eckart-Young theorem says the best rank-$r$ approximation of *any*
matrix is well-defined and computable via SVD. If the true weight update a full fine-tune
*would* have learned, $\Delta W_{\text{full}}$, happens to be well-approximated by a low-rank
matrix — i.e. its singular values decay quickly — then constraining $\Delta W$ to rank $r$
from the start loses little. Empirically (Hu et al., 2021; Aghajanyan et al., 2020 on
"intrinsic dimensionality") this turns out to be broadly true for pretrained language models:
the update needed to specialize a large pretrained model to a new task tends to live in a
much lower-dimensional space than the full parameter count suggests. We test this directly
below, not by asserting it, but by comparing an actual LoRA-trained update against the
*provably optimal* rank-$r$ approximation of a real fine-tuning delta.

### Rank choice and multi-rank adaptation

$r$ is the one real hyperparameter LoRA adds. Too small, and the adapter can't express the
needed update (underfits); large enough, and it approaches full fine-tuning's flexibility —
the rank-vs-performance curve is a genuine tradeoff curve, not a free lunch, which we'll measure
directly. In practice, different weight matrices (attention $W_Q$ vs. $W_V$, different layers)
often benefit from *different* ranks — a **multi-rank adaptation** strategy that spends the
parameter budget where it does the most good instead of applying one uniform $r$ everywhere.

### QLoRA: quantize the frozen part

Freezing $W_0$ means it's never updated, but it still occupies memory and must be read during
every forward/backward pass. **QLoRA** (Dettmers et al., 2023) observes that since $W_0$ is
frozen, it doesn't need full precision — it can be stored in **4-bit** quantized form (a much
coarser numeric representation), while the small trainable $A,B$ matrices stay in higher
precision (e.g. 16-bit) where precision actually matters for learning. This lets a 65B-parameter
model be fine-tuned on a single 48GB GPU, memory that would never fit a full-precision copy of
the same model plus its optimizer state. We describe this qualitatively rather than implement
4-bit quantization from scratch — the interesting *mathematics* here is still the low-rank
update, and 4-bit quantization is an orthogonal, purely numerical-precision trick (related to
[06.01](../06_Numerical_Methods/01_floating_point_arithmetic.ipynb)'s floating-point
representation questions, just taken further).

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import copy
import time
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
torch.set_num_threads(1)   # tiny tensors on CPU: thread-launch overhead dominates otherwise

## 👁️ Visual Intuition

Two things to see before any code: first, that $B=0$ initialization makes a LoRA layer
*provably* identical to its frozen base at the very start of training, only diverging as
training proceeds. Second, just how steep the parameter-count tradeoff is: for a realistic
hidden dimension, even a rank in the tens is a tiny fraction of the full matrix.

In [ ]:
class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear and adds a trainable low-rank update BA, scaled by alpha/r.
    B starts at exactly zero, so the wrapped layer starts out IDENTICAL to the base layer."""
    def __init__(self, base_linear, r, alpha=None):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad_(False)          # W0 is frozen: never updated by the optimizer
        d_out, d_in = base_linear.weight.shape
        self.r = r
        self.alpha = alpha if alpha is not None else r
        self.A = nn.Parameter(torch.randn(r, d_in) * (1.0 / d_in ** 0.5))
        self.B = nn.Parameter(torch.zeros(d_out, r))   # zero init -- delta_W = BA = 0 at start
    def forward(self, x):
        return self.base(x) + (self.alpha / self.r) * ((x @ self.A.T) @ self.B.T)
    def delta_weight(self):
        return (self.alpha / self.r) * self.B @ self.A

# --- Left panel: does a LoRA layer start out IDENTICAL to its frozen base, then diverge? ---
torch.manual_seed(0)
base_layer = nn.Linear(10, 10)
lora_layer = LoRALinear(base_layer, r=4)
x_demo = torch.randn(5, 10)
target_demo = torch.randn(5, 10)
opt_demo = torch.optim.Adam(lora_layer.parameters(), lr=0.05)

divergence_from_base = [torch.linalg.norm(lora_layer(x_demo) - base_layer(x_demo)).item()]
for _ in range(60):
    opt_demo.zero_grad()
    loss = ((lora_layer(x_demo) - target_demo) ** 2).mean()
    loss.backward()
    opt_demo.step()
    divergence_from_base.append(torch.linalg.norm(lora_layer(x_demo) - base_layer(x_demo)).item())

# --- Right panel: parameter count, full fine-tuning vs. LoRA, across a range of ranks ---
d_model = 512   # a realistic-scale hidden dimension (real LLMs: thousands)
ranks = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256])
full_params = d_model * d_model
lora_params = 2 * d_model * ranks

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

axes[0].plot(divergence_from_base, color="#4C72B0")
axes[0].axhline(0, color="gray", ls=":", lw=1)
axes[0].scatter([0], [divergence_from_base[0]], color="#C44E52", zorder=5, label=f"step 0: divergence = {divergence_from_base[0]:.4f}")
axes[0].set_xlabel("training step")
axes[0].set_ylabel("$\\|\\mathrm{LoRA}(x) - \\mathrm{base}(x)\\|$")
axes[0].set_title("B=0 init: LoRA starts EXACTLY at the base model")
axes[0].legend(fontsize=9)

axes[1].semilogy(ranks, lora_params, 'o-', color="#55A868", label="LoRA params: $2 d_{model} r$")
axes[1].axhline(full_params, color="#C44E52", ls="--", label=f"full fine-tune params: $d_{{model}}^2$ = {full_params:,}")
axes[1].set_xlabel("LoRA rank $r$")
axes[1].set_ylabel("trainable parameters (log scale)")
axes[1].set_title(f"Parameter count vs. rank ($d_{{model}}$={d_model})")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Divergence from base at step 0: {divergence_from_base[0]:.2e} (exactly zero, up to floating point)")
print(f"Divergence after 60 steps of training: {divergence_from_base[-1]:.4f}")
print(f"\nAt d_model={d_model}, full fine-tuning of one weight matrix = {full_params:,} parameters.")
for r in [8, 32, 128]:
    print(f"  LoRA rank {r:3d}: {2*d_model*r:,} parameters ({100*2*d_model*r/full_params:.2f}% of full, "
          f"{full_params/(2*d_model*r):.0f}x fewer)")

## 🐍 Implementation from Scratch

We build a small pretrain-then-adapt pipeline end to end: pretrain a tiny network on a base
task, then adapt it to a genuinely different downstream task two ways -- full fine-tuning of
one weight matrix, and LoRA at several ranks -- on the exact same data, exact same frozen
starting point. Then we connect directly back to [08.05](05_low_rank_approximations.ipynb): we
extract the *true* weight update full fine-tuning actually learned, compute its *provably
optimal* rank-$r$ approximation via truncated SVD (Eckart-Young), and compare that theoretical
ceiling against what gradient-descent-trained LoRA actually achieves at the same rank.

In [ ]:
torch.manual_seed(0)

# --- 1. A tiny network, pretrained on a BASE task ---
d_in, hidden, n_classes = 24, 128, 8
n_pretrain = 3000

class TinyNet(nn.Module):
    """fc2 (the middle layer) is what we adapt -- the same role a Q/V attention projection plays."""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(d_in, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, n_classes)
    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.fc3(x)

model = TinyNet()
X_pretrain = torch.randn(n_pretrain, d_in)
teacher_base = TinyNet()                      # a random "ground truth" defining the base task
with torch.no_grad():
    y_pretrain = teacher_base(X_pretrain).argmax(dim=1)

opt = torch.optim.Adam(model.parameters(), lr=0.01)
for _ in range(250):
    opt.zero_grad()
    loss = nn.functional.cross_entropy(model(X_pretrain), y_pretrain)
    loss.backward()
    opt.step()
print(f"Pretraining done: base-task accuracy = {(model(X_pretrain).argmax(1) == y_pretrain).float().mean().item():.4f}")

base_state = copy.deepcopy(model.state_dict())   # W0 for every layer, frozen from here on
W0_fc2 = model.fc2.weight.detach().clone()

# --- 2. A genuinely DIFFERENT downstream task (same input space, shifted decision boundary) ---
teacher_downstream = copy.deepcopy(teacher_base)
with torch.no_grad():
    teacher_downstream.fc3.weight += 0.6 * torch.randn_like(teacher_downstream.fc3.weight)

n_down, n_test = 2000, 1000
X_down, X_test = torch.randn(n_down, d_in), torch.randn(n_test, d_in)
with torch.no_grad():
    y_down = teacher_downstream(X_down).argmax(dim=1)
    y_test = teacher_downstream(X_test).argmax(dim=1)
    base_test_loss = nn.functional.cross_entropy(model(X_test), y_test).item()
    base_test_acc = (model(X_test).argmax(1) == y_test).float().mean().item()
print(f"\nUn-adapted base model on the downstream task: loss={base_test_loss:.3f}, acc={base_test_acc:.3f} "
      f"(chance level = 1/{n_classes} = {1/n_classes:.3f})")
print("Confirms a genuine task shift exists -- the base model, unmodified, is barely better than guessing.")

# --- 3. FULL fine-tuning: unfreeze ONLY fc2 (the layer LoRA will also target), train fully ---
model_full = copy.deepcopy(model)
model_full.load_state_dict(base_state)
for name, p in model_full.named_parameters():
    p.requires_grad_(name.startswith("fc2"))
opt_full = torch.optim.Adam([p for p in model_full.parameters() if p.requires_grad], lr=0.01)
for _ in range(300):
    opt_full.zero_grad()
    loss = nn.functional.cross_entropy(model_full(X_down), y_down)
    loss.backward()
    opt_full.step()

n_full_trainable = sum(p.numel() for n, p in model_full.named_parameters() if n.startswith("fc2"))
test_loss_full = nn.functional.cross_entropy(model_full(X_test), y_test).item()
test_acc_full = (model_full(X_test).argmax(1) == y_test).float().mean().item()
delta_W_full = model_full.fc2.weight.detach() - W0_fc2   # the TRUE weight update full fine-tuning learned
print(f"\nFULL fine-tuning (fc2 only): {n_full_trainable} trainable params, "
      f"test loss={test_loss_full:.4f}, test acc={test_acc_full:.4f}")

# --- 4. LoRA at several ranks, trained by ordinary gradient descent (NOT by SVD) ---
def train_lora(r, epochs=300, lr=0.03, seed=0):
    torch.manual_seed(seed)
    m = copy.deepcopy(model)
    m.load_state_dict(base_state)
    for p in m.parameters():
        p.requires_grad_(False)          # freeze EVERYTHING first
    m.fc2 = LoRALinear(m.fc2, r=r)        # only the new A, B inside fc2 are trainable
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.functional.cross_entropy(m(X_down), y_down)
        loss.backward()
        opt.step()
    test_loss = nn.functional.cross_entropy(m(X_test), y_test).item()
    test_acc = (m(X_test).argmax(1) == y_test).float().mean().item()
    n_trainable = sum(p.numel() for p in params)
    return test_loss, test_acc, n_trainable, m.fc2.delta_weight().detach()

# --- 5. The Eckart-Young "oracle": the PROVABLY OPTIMAL rank-r approximation of delta_W_full ---
U, S, Vt = torch.linalg.svd(delta_W_full)
def best_possible_rank_r(r):
    """08.05's Eckart-Young result, applied to a REAL fine-tuning weight update."""
    delta_r = U[:, :r] @ torch.diag(S[:r]) @ Vt[:r, :]
    m2 = copy.deepcopy(model)
    m2.load_state_dict(base_state)
    with torch.no_grad():
        m2.fc2.weight.copy_(W0_fc2 + delta_r)
        m2.fc2.bias.copy_(model_full.fc2.bias.detach())
    loss = nn.functional.cross_entropy(m2(X_test), y_test).item()
    acc = (m2(X_test).argmax(1) == y_test).float().mean().item()
    return loss, acc

print(f"\n{'rank':>5} {'LoRA params':>12} {'LoRA test acc':>14} {'EY oracle acc':>14}")
for r in [1, 2, 4, 8, 16, 32]:
    lora_loss, lora_acc, n_trainable, delta_lora = train_lora(r)
    oracle_loss, oracle_acc = best_possible_rank_r(r)
    print(f"{r:>5} {n_trainable:>12} {lora_acc:>14.4f} {oracle_acc:>14.4f}")

print(f"\nFull fine-tuning ({n_full_trainable} params) reaches test acc={test_acc_full:.4f}. Gradient-descent")
print("LoRA tracks the Eckart-Young oracle's trend closely -- both need similar ranks to approach")
print("full fine-tuning, confirming LoRA finds close to the best possible rank-r update.")

## 🤖 Why This Matters for AI

The tiny experiment above scales directly: at GPT-3's actual dimensions, the same
$W=W_0+BA$ mechanism is why LoRA can fine-tune a 175-billion-parameter model while updating
only a few million numbers. We compute this scaling explicitly (matching the parameter-count
claim from Hu et al., 2021), then quantify **QLoRA**'s (Dettmers et al., 2023) added twist:
keep $W_0$ frozen (as LoRA already does) but store it in 4-bit precision, since a matrix that's
never updated doesn't need full-precision storage.

In [ ]:
# 1) Scaling the tiny experiment above to REAL LLM dimensions (matches Hu et al. 2021's own numbers)
d_model, n_layers, r = 12288, 96, 4    # GPT-3 175B's actual hidden dim and layer count; r=4 from the paper
full_params_per_matrix = d_model * d_model
lora_params_per_matrix = 2 * d_model * r
n_matrices = n_layers * 2              # LoRA applied to W_q and W_v in every layer (paper's main config)

total_full = full_params_per_matrix * n_matrices
total_lora = lora_params_per_matrix * n_matrices
total_model_params = 175_000_000_000

print(f"GPT-3 scale: d_model={d_model}, {n_layers} layers, LoRA rank r={r}, applied to W_q & W_v")
print(f"  full fine-tuning of JUST Wq,Wv everywhere: {total_full:,} params ({total_full/1e9:.1f}B)")
print(f"  LoRA for those same matrices:               {total_lora:,} params ({total_lora/1e6:.1f}M)")
print(f"  reduction vs. fine-tuning Wq,Wv alone: {total_full/total_lora:,.0f}x fewer trainable parameters")
print(f"  LoRA params as a fraction of the FULL 175B model: {100*total_lora/total_model_params:.4f}%")
print(f"  reduction vs. the ENTIRE 175B model: {total_model_params/total_lora:,.0f}x")
print(f"  (Hu et al. 2021 report ~10,000x fewer trainable parameters vs. the FULL model --")
print(f"   our estimate of the same ratio, {total_model_params/total_lora:,.0f}x, matches their claim;")
print(f"   the {total_full/total_lora:,.0f}x figure above answers a narrower question -- how much")
print(f"   LoRA saves relative to fully fine-tuning only Wq/Wv, rather than the whole model.)")

# 2) QLoRA: quantize the FROZEN base weights to 4-bit, keep LoRA's A, B in higher precision.
# We don't implement 4-bit quantization here (an orthogonal numerical-precision technique --
# see 06.01 on floating-point representations) -- but the memory arithmetic is straightforward
# and is QLoRA's actual headline result.
print("\n--- QLoRA's memory arithmetic (Dettmers et al., 2023) ---")
n_params_65b = 65_000_000_000
gb_fp16 = n_params_65b * 2 / 1e9      # 2 bytes/param at 16-bit precision
gb_4bit = n_params_65b * 0.5 / 1e9    # 0.5 bytes/param at 4-bit precision
print("65B model (the QLoRA paper's headline example):")
print(f"  stored at 16-bit: {gb_fp16:.1f} GB (does not fit on a single 48GB GPU)")
print(f"  stored at 4-bit:  {gb_4bit:.1f} GB (fits on a single 48GB GPU, room left for activations + adapters)")
print("\nQLoRA backpropagates through the frozen 4-bit base to train full-precision LoRA adapters --")
print("the LOW-RANK part of the math is unchanged; only the STORAGE precision of the frozen W0")
print("changes, which is why QLoRA needs no new derivation, only a new number format.")

## 🏋️ Exercises

### Warm-up
1. For $d=4$, $r=1$, $B=(1,2,0,-1)^\top$, $A=(2,0,1,1)$, compute $\Delta W = BA$ by hand
   (a $4\times4$ matrix) and confirm $\text{rank}(\Delta W)=1$. Count the parameters in $B$ and
   $A$ combined versus the $16$ entries of a full $4\times4$ matrix, and verify with
   `torch.outer` or plain matrix multiplication.

### Practice
2. In the Implementation section, rerun `train_lora` with `alpha` set to `4*r` instead of `r`
   (i.e. `LoRALinear(..., r=r, alpha=4*r)`) for a couple of ranks, keeping the same learning
   rate. Explain what changes about the effective update `(alpha/r) * B @ A`, and why LoRA
   papers treat $\alpha$ and $r$ as two separate hyperparameters rather than folding $\alpha$
   into the learning rate directly (hint: what happens to the *initial* gradient magnitude on
   $B$ if you increase $\alpha$ alone, before $B$ has moved from zero?).

### Challenge
3. Add a SECOND downstream task (a new `teacher_downstream_2`, built the same way as
   `teacher_downstream` but with an independent random perturbation) and train separate LoRA
   adapters for each task on top of the SAME frozen base model. Confirm that swapping which
   adapter is attached to the frozen base changes which task the model is good at, without ever
   touching the base weights. Explain in a sentence why this "swap the adapter" property is
   valuable for serving many fine-tuned variants of one base model in production.

---

## 📚 Further Reading
- Hu et al., ["LoRA: Low-Rank Adaptation of Large Language Models"](https://arxiv.org/abs/2106.09685)
- Dettmers et al., ["QLoRA: Efficient Finetuning of Quantized LLMs"](https://arxiv.org/abs/2305.14314)
- Aghajanyan, Zettlemoyer & Gupta, ["Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning"](https://arxiv.org/abs/2012.13255)
- Hugging Face PEFT documentation, ["LoRA conceptual guide"](https://huggingface.co/docs/peft/conceptual_guides/lora)

---

*Next notebook: [Measure Theory Essentials](../09_Advanced_Probability/01_measure_theory_essentials.ipynb)*